In [2]:
import pandas as pd
import numpy as np

DATA_PATH = r"D:\Course\python\energy forecasting\data\processed\final_energy_forecasting_dataset.csv"

df = pd.read_csv(DATA_PATH)

df["time"] = pd.to_datetime(df["time"])
df = df.sort_values("time").reset_index(drop=True)

print("Dataset shape:", df.shape)
df.head()

Dataset shape: (50225, 27)


,time,load,solar,wind,wind_onshore,wind_offshore,hour,day_of_week,day_of_month,month,...,weather_code,temperature_lag_1,humidity_lag_1,temperature_lag_24,humidity_lag_24,load_lag_1,load_lag_24,load_lag_168,rolling_mean_24,rolling_std_24
0,2015-01-08 07:00:00+00:00,68569.0,57.0,18039.0,17503.0,536.0,7,3,8,1,...,71,1.4,86.0,0.4,89.0,65447.0,65964.0,41133.0,60983.125000,8202.879533
1,2015-01-08 08:00:00+00:00,68599.0,446.0,18177.0,17649.0,527.0,8,3,8,1,...,51,1.7,85.0,0.8,93.0,68569.0,66400.0,42963.0,61074.750000,8277.953655
2,2015-01-08 09:00:00+00:00,69484.0,1083.0,18094.0,17566.0,529.0,9,3,8,1,...,71,1.7,88.0,1.3,95.0,68599.0,67746.0,45088.0,61147.166667,8346.173123
3,2015-01-08 10:00:00+00:00,70635.0,1738.0,17924.0,17400.0,524.0,10,3,8,1,...,51,1.7,92.0,2.3,96.0,69484.0,68507.0,47013.0,61235.833333,8438.553053
4,2015-01-08 11:00:00+00:00,69962.0,2062.0,17249.0,16774.0,474.0,11,3,8,1,...,3,2.4,92.0,3.3,96.0,70635.0,68100.0,48159.0,61313.416667,8512.639773


In [6]:
print("Start date:", df["time"].min())
print("End date:", df["time"].max())
print("Total rows:", len(df))

Start date: 2015-01-08 07:00:00+00:00
End date: 2020-09-30 23:00:00+00:00
Total rows: 50225


In [7]:
last_30_days = df[df["time"] >= df["time"].max() - pd.Timedelta(days=30)]

print("Last 30 days shape:", last_30_days.shape)

last_30_days.to_csv("test_last_30_days.csv", index=False)

Last 30 days shape: (721, 27)


In [8]:
last_90_days = df[df["time"] >= df["time"].max() - pd.Timedelta(days=90)]

print("Last 90 days shape:", last_90_days.shape)

last_90_days.to_csv("test_last_90_days.csv", index=False)

Last 90 days shape: (2161, 27)


In [9]:
window_size = 60 * 24  # 60 days hourly

start_idx = np.random.randint(0, len(df) - window_size)

random_window = df.iloc[start_idx:start_idx + window_size]

print("Random window shape:", random_window.shape)

random_window.to_csv("test_random_60_days.csv", index=False)

Random window shape: (1440, 27)


In [10]:
LOOKBACK = 720

def create_lookback_windows(df, num_samples=5):
    samples = []
    
    for i in range(num_samples):
        start_idx = np.random.randint(0, len(df) - LOOKBACK)
        sample = df.iloc[start_idx:start_idx + LOOKBACK]
        
        filename = f"test_lookback_{i+1}.csv"
        sample.to_csv(filename, index=False)
        
        samples.append(filename)
        
    return samples

files_created = create_lookback_windows(df, num_samples=5)

print("Created files:", files_created)

Created files: ['test_lookback_1.csv', 'test_lookback_2.csv', 'test_lookback_3.csv', 'test_lookback_4.csv', 'test_lookback_5.csv']


In [11]:
LOOKBACK = 720
STEP = 168  # Move by 1 week

rolling_files = []

for start in range(0, len(df) - LOOKBACK, STEP):
    sample = df.iloc[start:start + LOOKBACK]
    
    filename = f"rolling_test_{start}.csv"
    sample.to_csv(filename, index=False)
    
    rolling_files.append(filename)

print("Total rolling test files:", len(rolling_files))

Total rolling test files: 295


In [12]:
LOOKBACK = 720
MAX_LAG = 168

REQUIRED_ROWS = LOOKBACK + MAX_LAG

def create_valid_lookback(df, num_samples=5):
    samples = []

    for i in range(num_samples):
        start_idx = np.random.randint(0, len(df) - REQUIRED_ROWS)
        sample = df.iloc[start_idx:start_idx + REQUIRED_ROWS]

        filename = f"test_valid_lookback_{i+1}.csv"
        sample.to_csv(filename, index=False)

        samples.append(filename)

    return samples

files = create_valid_lookback(df, 3)
print(files)

['test_valid_lookback_1.csv', 'test_valid_lookback_2.csv', 'test_valid_lookback_3.csv']


Forecast dataset saved: (720, 27)
Evaluation dataset saved: (744, 27)
7D evaluation dataset saved: (888, 27)
30D evaluation dataset saved: (1440, 27)


In [17]:
import pandas as pd

df = pd.read_csv(DATA_PATH)

df["time"] = pd.to_datetime(df["time"])
df = df.sort_values("time").reset_index(drop=True)

LOOKBACK = 720

# Add 200 extra rows for safe lag drop
forecast_df = df.tail(LOOKBACK + 200).copy()

forecast_df.to_csv("forecast_test.csv", index=False)

print("New forecast dataset shape:", forecast_df.shape)

New forecast dataset shape: (920, 27)


In [3]:
import pandas as pd

df = pd.read_csv(DATA_PATH)

df["time"] = pd.to_datetime(df["time"])
df = df.sort_values("time").reset_index(drop=True)

LOOKBACK = 720

# Need enough rows for 30D evaluation
rows_needed = LOOKBACK + 720 + 200   # extra buffer for dropna

eval_30d_df = df.tail(rows_needed).copy()
eval_30d_df.to_csv("evaluation_test_30d_big.csv", index=False)

print("Saved 30D evaluation dataset:", eval_30d_df.shape)

Saved 30D evaluation dataset: (1640, 27)


In [ ]:

# link https://energy-forecast-api-sfrz.onrender.com/run_model